In [0]:
table_name = "dim_providers"
silver_table_fqn = f"hdfc_ergo.hdfc_silver.{table_name}"

df = spark.read.table("hdfc_ergo.hdfc_bronze.dim_providers")
display(df.limit(5))

In [0]:
from pyspark.sql.functions import col, upper, trim, lower, regexp_replace, when, to_date, row_number, abs as spark_abs
from pyspark.sql.window import Window

# 1. Standardize Text: UPPER(TRIM())
text_cols = ["provider_id", "registration_number", "provider_name", "provider_type", 
             "specialty", "qualification", "city", "state", "pincode", 
             "license_number", "license_status", "network_status", 
             "languages_spoken", "emr_system"]
for c in text_cols:
    df = df.withColumn(c, upper(trim(col(c))))

df = df.withColumn("email", lower(trim(col("email"))))

# 2. Clean Phone (Remove non-numeric)
df = df.withColumn("phone", regexp_replace(col("phone"), "[^0-9]", ""))

# 3. Boolean Mapping
for bool_col in ["board_certified", "capitation_flag", "accepting_new_patients", "telehealth_enabled", "hipaa_compliant"]:
    df = df.withColumn(bool_col, 
                       when(upper(trim(col(bool_col))).isin("YES", "1", "TRUE"), True)
                       .when(upper(trim(col(bool_col))).isin("NO", "0", "FALSE"), False)
                       .otherwise(None).cast("boolean"))

# 4. Safe Cast Integers
df = df.withColumn("years_of_experience", trim(col("years_of_experience")).cast("int"))

# 5. Precision Optimization (Safe Decimal Cast to avoid "Unlimited" or text crashes)
for dec_col in ["quality_score", "patient_satisfaction", "average_consultation_fee"]:
    is_valid_decimal = trim(col(dec_col)).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
    df = df.withColumn(dec_col, when(is_valid_decimal, trim(col(dec_col)).cast("decimal(15,2)")).otherwise(None))
    if dec_col == "average_consultation_fee":
        df = df.withColumn(dec_col, spark_abs(col(dec_col))) # ABS for financials

# 6. Safe Cast Dates
for date_col in ["record_created_date", "record_modified_date"]:
    df = df.withColumn(date_col, to_date(trim(col(date_col))))

# 7. Deduplication (PARTITION BY registration_number)
dedup_window = Window.partitionBy("registration_number").orderBy(col("_ingested_at").desc())
df = df.withColumn("_dq_is_deduped", row_number().over(dedup_window))
df = df.filter(col("_dq_is_deduped") == 1).drop("_dq_is_deduped")

# 8. Drop audit columns and source surrogate keys
df = df.drop("_ingested_at", "_source_file", "provider_sk", "facility_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(silver_table_fqn)

print(f"✅ Successfully wrote {table_name} to Silver layer.")